In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_calls_cti(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "CallsId    string,"
        # "SourceSystem  string,"
        # "IngestionTimestamp	timestamp,"
        # "BatchId  string,"
        # "CallsKey string,"
        # "CallType string,"
        # "CallDate  timestamp,"
        # "Selection string,"
        # "Campaign string,"
        # "NumberPhone  string,"
        # "InputNumber string,"
        # "From string,"
        # "To date,"
        # "Agent string,"
        # "Context string,"
        # "Channel string,"
        # "Duration int,"
        # "Billsec int,"
        # "Disposition string,"
        # "FromCall string,"
        # "MaxRingTime int,"
        # "OverflowToPhone string,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Calls"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "cti",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ===============  CallBackService_IVR ==============

        callbackservice_ivr_extra_filters = None

        if callbackservice_ivr_extra_filters is not None:
            callbackservice_ivr_extra_filters.append(date_filter)
            callbackservice_ivr_final_filters = " AND ".join(
                callbackservice_ivr_extra_filters
            )
        else:
            callbackservice_ivr_final_filters = date_filter

        callbackservice_ivr_selected_fields = [
            "UniqueID",
            "ID",
            "callDT",
            "Selection",
            "Campaign",
            "OriginalNumber",
            "InputNumber",
            "BusinessDate",
        ]

        callbackservice_ivr = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_callbackservice_ivr",
            callbackservice_ivr_final_filters,
            callbackservice_ivr_selected_fields,
        )

        # ================== ForcedCalls  ===============

        forced_calls_extra_filters = None

        if forced_calls_extra_filters is not None:
            forced_calls_extra_filters.append(date_filter)
            forced_calls_final_filters = " AND ".join(forced_calls_extra_filters)
        else:
            forced_calls_final_filters = date_filter

        forced_calls_selected_fields = [
            "UniqueID",
            "RecordNum",
            "Calldate",
            "Agent",
            "To",
            "From",
            "Context",
            "Channel",
            "Duration",
            "Billsec",
            "Disposition",
            "FromCall",
            "BusinessDate",
        ]

        forced_calls = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_forcedcalls",
            forced_calls_final_filters,
            forced_calls_selected_fields,
        )

        # ================= RedirectCalls ================

        redirect_calls_extra_filters = None

        if redirect_calls_extra_filters is not None:
            redirect_calls_extra_filters.append(date_filter)
            redirect_calls_final_filters = " AND ".join(redirect_calls_extra_filters)
        else:
            redirect_calls_final_filters = date_filter

        redirect_calls_selected_fields = [
            "UniqueID",
            "DateTime",
            "Campaign",
            "CustomerPhone",
            "BusinessDate",
        ]

        redirect_calls = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_redirectcalls",
            redirect_calls_final_filters,
            redirect_calls_selected_fields,
        )

        # ================ RedirectOverflowCalls ==========

        redirect_over_flow_calls_extra_filters = None

        if redirect_over_flow_calls_extra_filters is not None:
            redirect_over_flow_calls_extra_filters.append(date_filter)
            redirect_over_flow_calls_final_filters = " AND ".join(
                redirect_over_flow_calls_extra_filters
            )
        else:
            redirect_over_flow_calls_final_filters = date_filter

        redirect_over_flow_calls_selected_fields = [
            "UniqueID",
            "Datetime",
            "CampaignName",
            "CustomerPhone",
            "MaxRingTime",
            "OverflowTo",
            "BusinessDate",
        ]

        redirect_over_flow_calls = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_redirectoverflowcalls",
            redirect_over_flow_calls_final_filters,
            redirect_over_flow_calls_selected_fields,
        )

        # ================ callbackservice_ivr_df  construction ===============
        callbackservice_ivr_primary_key_fields = [
            callbackservice_ivr.UniqueID,
            callbackservice_ivr.ID,
            lit("cti"),
        ]

        callbackservice_ivr_df = callbackservice_ivr.withColumn(
            "CallsId",
            sha2(concat_ws(";", *callbackservice_ivr_primary_key_fields), 256),
        ).withColumn(
            "CallsKey",
            concat_ws(";", callbackservice_ivr.UniqueID, callbackservice_ivr.ID),
        )

        result_callbackservice_ivr_df = callbackservice_ivr_df.select(
            callbackservice_ivr_df.CallsId.cast("string").alias("CallsId"),
            lit("cti").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            callbackservice_ivr_df.CallsKey.cast("string").alias("CallsKey"),
            lit(None).alias("CallType"),
            callbackservice_ivr_df.callDT.cast("timestamp").alias("CallDate"),
            callbackservice_ivr_df.Selection,
            callbackservice_ivr_df.Campaign,
            callbackservice_ivr_df.OriginalNumber.alias("NumberPhone"),
            callbackservice_ivr_df.InputNumber,
            lit(None).alias("From"),
            lit(None).alias("To"),
            lit(None).alias("Agent"),
            lit(None).alias("Context"),
            lit(None).alias("Channel"),
            lit(None).alias("Duration"),
            lit(None).alias("Billsec"),
            lit(None).alias("Disposition"),
            lit(None).alias("FromCall"),
            lit(None).alias("MaxRingTime"),
            lit(None).alias("OverflowToPhone"),
            callbackservice_ivr_df.BusinessDate.alias("BusinessDate"),
        )

        # ================ ForcedCalls_df construction ===============

        forced_calls_primary_key_fields = [
            forced_calls.UniqueID,
            forced_calls.RecordNum,
            lit("cti"),
        ]

        forced_calls_df = forced_calls.withColumn(
            "CallsId", sha2(concat_ws(";", *forced_calls_primary_key_fields), 256)
        ).withColumn(
            "CallsKey", concat_ws(";", forced_calls.UniqueID, forced_calls.RecordNum)
        )

        result_forced_calls_df = forced_calls_df.select(
            forced_calls_df.CallsId.cast("string").alias("CallsId"),
            lit("cti").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            forced_calls_df.CallsKey.alias("CallsKey"),
            lit(None).alias("CallType"),
            forced_calls_df.Calldate.cast("timestamp").alias("CallDate"),
            lit(None).alias("Selection"),
            lit(None).alias("Campaign"),
            lit(None).alias("NumberPhone"),
            lit(None).alias("InputNumber"),
            lit(None).alias("From"),
            lit(None).alias("To"),
            lit(None).alias("Agent"),
            lit(None).alias("Context"),
            lit(None).alias("Channel"),
            lit(None).alias("Duration"),
            lit(None).alias("Billsec"),
            lit(None).alias("Disposition"),
            lit(None).alias("FromCall"),
            lit(None).alias("MaxRingTime"),
            lit(None).alias("OverflowToPhone"),
            forced_calls_df.BusinessDate.alias("BusinessDate"),
        )

        # ================ redirect_calls_df construction =========

        redirect_calls_primary_key_fields = [redirect_calls.UniqueID, lit("cti")]
        redirect_calls_df = redirect_calls.withColumn(
            "CallsId", sha2(concat_ws(";", *redirect_calls_primary_key_fields), 256)
        )

        result_redirect_calls_df = redirect_calls_df.select(
            redirect_calls_df.CallsId.cast("string").alias("CallsId"),
            lit("cti").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            redirect_calls_df.UniqueID.cast("string").alias("CallsKey"),
            lit(None).alias("CallType"),
            redirect_calls_df.DateTime.cast("timestamp").alias("CallDate"),
            lit(None).alias("Selection"),
            redirect_calls_df.Campaign.cast("string").alias("Campaign"),
            redirect_calls_df.CustomerPhone.alias("NumberPhone"),
            lit(None).alias("InputNumber"),
            lit(None).alias("From"),
            lit(None).alias("To"),
            lit(None).alias("Agent"),
            lit(None).alias("Context"),
            lit(None).alias("Channel"),
            lit(None).alias("Duration"),
            lit(None).alias("Billsec"),
            lit(None).alias("Disposition"),
            lit(None).alias("FromCall"),
            lit(None).alias("MaxRingTime"),
            lit(None).alias("OverflowToPhone"),
            redirect_calls_df.BusinessDate.alias("BusinessDate"),
        )

        # ==================  redirect_over_flow_calls_df construction ==============

        redirect_over_flow_calls_primary_key_fields = [
            redirect_over_flow_calls.UniqueID,
            lit("cti"),
        ]

        redirect_over_flow_calls_df = redirect_over_flow_calls.withColumn(
            "CallsId",
            sha2(concat_ws(";", *redirect_over_flow_calls_primary_key_fields), 256),
        )

        result_redirect_over_flow_calls_df = redirect_over_flow_calls_df.select(
            redirect_over_flow_calls_df.CallsId.cast("string").alias("CallsId"),
            lit("cti").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            redirect_over_flow_calls_df.UniqueID.alias("CallsKey"),
            lit(None).alias("CallType"),
            redirect_over_flow_calls_df.Datetime.cast("timestamp").alias("CallDate"),
            lit(None).alias("Selection"),
            redirect_over_flow_calls.CampaignName.alias("Campaign"),
            redirect_over_flow_calls_df.CustomerPhone.alias("NumberPhone"),
            lit(None).alias("InputNumber"),
            lit(None).alias("From"),
            lit(None).alias("To"),
            lit(None).alias("Agent"),
            lit(None).alias("Context"),
            lit(None).alias("Channel"),
            lit(None).alias("Duration"),
            lit(None).alias("Billsec"),
            lit(None).alias("Disposition"),
            lit(None).alias("FromCall"),
            redirect_over_flow_calls_df.MaxRingTime.alias("MaxRingTime"),
            redirect_over_flow_calls_df.OverflowTo.alias("OverflowToPhone"),
            redirect_over_flow_calls_df.BusinessDate.alias("BusinessDate"),
        )

        calls_df = (
            result_forced_calls_df.union(result_redirect_calls_df)
            .union(result_callbackservice_ivr_df)
            .union(result_redirect_over_flow_calls_df)
        )

        # ================== Append ==================

        calls_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_calls_cti(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)